In [1]:
  from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load Dataset
df = pd.read_csv("/content/drive/MyDrive/gdghack/data/new_simon.csv")
df = df[['Sentence', 'Category']]

# Encode Labels
labels = {label: i for i, label in enumerate(df['Category'].unique())}
df['label'] = df['Category'].map(labels)

# Train-Test Split
train_texts, val_texts, train_labels, val_labels = train_test_split(df['Sentence'].tolist(), df['label'].tolist(), test_size=0.2)

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Convert to Dataset Class
def convert_to_dataset(encodings, labels):
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })

train_dataset = convert_to_dataset(train_encodings, train_labels)
val_dataset = convert_to_dataset(val_encodings, val_labels)

# Load Pre-trained Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(labels))

# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to=[]  # Disable wandb
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train Model
trainer.train()

# Save Model Locally
model.save_pretrained("./bert-text-classification")
tokenizer.save_pretrained("./bert-text-classification")

# Save Model to Google Drive
google_drive_path = "/content/drive/MyDrive/gdghack/bert-text-classification"
model.save_pretrained(google_drive_path)
tokenizer.save_pretrained(google_drive_path)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.669381
2,No log,0.374417
3,No log,0.304452


('/content/drive/MyDrive/gdghack/bert-text-classification/tokenizer_config.json',
 '/content/drive/MyDrive/gdghack/bert-text-classification/special_tokens_map.json',
 '/content/drive/MyDrive/gdghack/bert-text-classification/vocab.txt',
 '/content/drive/MyDrive/gdghack/bert-text-classification/added_tokens.json')

In [ ]:
import re
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Load label map (same as training)
labels = {'Work': 0, 'Personal': 1, 'Miscellaneous': 2}
id2label = {v: k for k, v in labels.items()}

# Load trained model
model_path = "/content/drive/MyDrive/gdghack/bert-text-classification"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.eval()

# Example paragraph
paragraph = """
We need to submit assignments to the college.
"""

# Split paragraph into sentences (regex-based)
sentences = re.split(r'(?<=[.!?]) +', paragraph.strip())

# Classify each sentence
for sent in sentences:
    inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=1).item()
        print(f"📝 Sentence: {sent.strip()}\n📂 Predicted Category: {id2label[pred_id]}\n")



📝 Sentence: We need to submit assignments to the college.
📂 Predicted Category: Work



#WITH PERFORMANCE

In [3]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# Load Dataset
df = pd.read_csv("/content/drive/MyDrive/gdghack/data/new_simon.csv")
df = df[['Sentence', 'Category']]

# Encode Labels
labels = {label: i for i, label in enumerate(df['Category'].unique())}
df['label'] = df['Category'].map(labels)

# Train-Test Split
train_texts, val_texts, train_labels, val_labels = train_test_split(df['Sentence'].tolist(), df['label'].tolist(), test_size=0.2)

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Convert to Dataset Class
def convert_to_dataset(encodings, labels):
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })

train_dataset = convert_to_dataset(train_encodings, train_labels)
val_dataset = convert_to_dataset(val_encodings, val_labels)

# Load Pre-trained Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(labels))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    report_to=[]  # Disable wandb
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Train Model
trainer.train()
# Evaluate and print metrics
metrics = trainer.evaluate()
print("Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# Save Model Locally
model.save_pretrained("./bert-gdg-classification")
tokenizer.save_pretrained("./bert-gdg-classification")

# Save Model to Google Drive
google_drive_path = "/content/drive/MyDrive/gdghack/bert-gdg-classification"
model.save_pretrained(google_drive_path)
tokenizer.save_pretrained(google_drive_path)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.691043,0.820513,0.827048,0.820513,0.819100
2,No log,0.459675,0.858974,0.872363,0.858974,0.858932
3,No log,0.335166,0.884615,0.885425,0.884615,0.884818
4,No log,0.320031,0.897436,0.900377,0.897436,0.898284


Evaluation Metrics:
eval_loss: 0.3200
eval_accuracy: 0.8974
eval_precision: 0.9004
eval_recall: 0.8974
eval_f1: 0.8983
eval_runtime: 0.2171
eval_samples_per_second: 359.3460
eval_steps_per_second: 46.0700
epoch: 4.0000


('/content/drive/MyDrive/gdghack/bert-gdg-classification/tokenizer_config.json',
 '/content/drive/MyDrive/gdghack/bert-gdg-classification/special_tokens_map.json',
 '/content/drive/MyDrive/gdghack/bert-gdg-classification/vocab.txt',
 '/content/drive/MyDrive/gdghack/bert-gdg-classification/added_tokens.json')

#Testing

In [6]:
import re
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Load label map (same as training)
labels = {'Work': 0, 'Personal': 1, 'Miscellaneous': 2}
id2label = {v: k for k, v in labels.items()}

# Load trained model
model_path = "/content/drive/MyDrive/gdghack/bert-gdg-classification"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.eval()

# Example paragraph
paragraph = """
I was talking to them about the plans we discussed earlier. It’s still unclear how things will turn out, but everyone seemed interested. We might need to make some changes depending on how things go. I’ll follow up once I get a clearer picture."""

# Split paragraph into sentences (regex-based)
sentences = re.split(r'(?<=[.!?]) +', paragraph.strip())

# Classify each sentence
for sent in sentences:
    inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=1).item()
        print(f"📝 Sentence: {sent.strip()}\n📂 Predicted Category: {id2label[pred_id]}\n")



📝 Sentence: I was talking to them about the plans we discussed earlier.
📂 Predicted Category: Work

📝 Sentence: It’s still unclear how things will turn out, but everyone seemed interested.
📂 Predicted Category: Work

📝 Sentence: We might need to make some changes depending on how things go.
📂 Predicted Category: Work

📝 Sentence: I’ll follow up once I get a clearer picture.
📂 Predicted Category: Work



In [12]:
import re
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification

# Load model and tokenizer
model_path = "/content/drive/MyDrive/gdghack/bert-gdg-classification"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.eval()

# Label mapping
labels = {'Work': 0, 'Personal': 1, 'Miscellaneous': 2}
id2label = {v: k for k, v in labels.items()}

# Paragraph to analyze
paragraph = """
Need to call grandma about her birthday gift, but I also spent half the morning rearranging my bookshelf. I think I'll take the dog for a walk later, and maybe order pizza tonight. My cousin messaged about a weekend plan—need to check that.
"""

# Split into sentences
sentences = re.split(r'(?<=[.!?]) +', paragraph.strip())

# Set confidence threshold
CONFIDENCE_THRESHOLD = 0.6  # You can adjust this

# Classify and filter
for sent in sentences:
    inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        confidence, pred_id = torch.max(probs, dim=1)
        confidence = confidence.item()
        pred_id = pred_id.item()

        if confidence >= CONFIDENCE_THRESHOLD:
            print(f"📝 Sentence: {sent.strip()}")
            print(f"📂 Predicted Category: {id2label[pred_id]}")
            print(f"🔢 Confidence: {confidence:.2f}\n")
        else:
            print(f"⚪ Skipped (low confidence - {confidence:.2f}): {sent.strip()}\n")


📝 Sentence: Need to call grandma about her birthday gift, but I also spent half the morning rearranging my bookshelf.
📂 Predicted Category: Personal
🔢 Confidence: 0.88

📝 Sentence: I think I'll take the dog for a walk later, and maybe order pizza tonight.
📂 Predicted Category: Personal
🔢 Confidence: 0.81

⚪ Skipped (low confidence - 0.49): My cousin messaged about a weekend plan—need to check that.

